# Setup

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import json, glob, os, re
import numpy as np
from matplotlib.patches import Patch, Rectangle
from matplotlib.colors import TwoSlopeNorm, LogNorm
from matplotlib.lines import Line2D
from matplotlib.axes import Axes
from scipy.stats import ttest_rel, pearsonr
from scipy.io import loadmat
import os, sys, configparser

settings_parser = configparser.ConfigParser()
settings_parser.read("localsettings.ini")
MAIN_DATA_FOLDER = settings_parser.get("global", "data_path")
MIENC_PATH = settings_parser.get("global", "mienc_path")

bands = {
    i: b
    for i, b in enumerate(
        [
            [1.0, 4.0],
            [4.0, 8.0],
            [8.0, 12.0],
            [12.0, 30.0],
            [30.0, 44.0],
        ],
        start=1,
    )
}

band_names = {
    i: b
    for i, b in enumerate(
        ["delta", "theta", "alpha", "beta", "gamma"],
        start=1,
    )
}

band_fancy_names = {
    i: b
    for i, b in enumerate(
        [r"$\delta$", r"$\theta$", r"$\alpha$", r"$\beta$", r"$\gamma$"],
        start=1,
    )
}



def fs_band(band):
    return int(bands[band][1] * 2 * 1.125 + 0.5)

nbins = [10, 13, 14, 20, 23]

# bands = ["delta", "theta", "alpha", "beta", "gamma"]
# band_names = [r"$\delta$", r"$\theta$", r"$\alpha$", r"$\beta$", r"$\gamma$"]
sns.set_theme("paper", "ticks")
sns.set_palette("colorblind")

cache_dir = os.path.join(MAIN_DATA_FOLDER, "cache")
RESULTS_FOLDER = os.path.join(MAIN_DATA_FOLDER, "Results")
FIGURES_FOLDER = os.path.join(MAIN_DATA_FOLDER, "Figures")
if not os.path.isdir(FIGURES_FOLDER):
    os.mkdir(FIGURES_FOLDER)
if not os.path.isdir(os.path.join(RESULTS_FOLDER, "localised")):
    os.mkdir(os.path.join(RESULTS_FOLDER, "localised"))

sys.path.append(MIENC_PATH)
from mienc import Corrector

In [ ]:
def coloured_box_kwargs(colour):
    return {
        "boxprops": {"color": colour},
        "widths": 0.6,
        "whiskerprops": {"color": colour},
        "flierprops": {"markeredgecolor": colour, "marker": "+"},
        "medianprops": {"lw": 1.6, "color": colour},
        "capprops": {"color": colour},
    }

In [ ]:
def boxplot(
    x, series, colours, percents, ax=None, ttest=False, variant="Holm-Bonferroni", legend=False
):
    if ax is not None:
        plt.sca(ax)
    STRETCH = len(series)
    OFFSET = -STRETCH / 2
    handles = []
    labels = []

    for data, colour in zip(series, colours):
        plt.boxplot(
            series[data],
            positions=np.arange(len(x)) * STRETCH + OFFSET,
            **coloured_box_kwargs(colour),
        )
        handles.append(Patch(facecolor="white", edgecolor=colour))
        labels.append(data)
        OFFSET += 1

    if ttest:
        if hasattr(ttest, "__len__") and len(ttest) == 2:
            first, second = ttest
        elif len(series) == 2:
            first, second = series.keys()
        else:
            raise ValueError(ttest)
        significance = np.array(
            [
                ttest_rel(a, b, alternative="greater").pvalue
                for a, b in zip(series[first], series[second])
            ]
        )
        if variant == "Bonferroni":
            threshold = 0.05 / len(x)
        else:
            moving_threshold = 0.05 / (len(x) - np.arange(len(x)))
            significance_sorter = np.argsort(significance)
            above_threshold = significance[significance_sorter] > moving_threshold
            if above_threshold.any():
                first_above = np.min(np.where(above_threshold))
                threshold = significance[significance_sorter[first_above]]
            else:
                threshold = 0.05

        markers = significance < threshold
        position = min(((max(percents) / 100 if percents else 0) + plt.ylim()[1]) / 2, 0.2)
        ast = plt.scatter(
            np.arange(len(x))[markers] * STRETCH - OFFSET / 2,
            np.full(np.sum(markers), position, dtype=float),
            marker="*",
            color="k",
            s=40,
        )
        handles.append(ast)
        labels.append("Significative\ndifference")

    for percent, style in zip(percents, ["solid", "dashed", "dashdot", "dotted"]):
        plt.hlines(
            percent / 100,
            -STRETCH,
            (len(x) - 0.5) * STRETCH,
            "r",
            style,
            lw=1,
            zorder=1,
        )
        handles.append(Line2D([0], [0], lw=1, color="r", linestyle=style))
        labels.append(f"{percent}%")

    tickPos = np.arange(len(x)) * STRETCH - 0.5
    plt.xticks(tickPos, x.values() if isinstance(x, dict) else x, rotation=45, ha="right", rotation_mode="anchor")
    plt.xlim((-0.8 * STRETCH, (len(x) - 0.6) * STRETCH))
    ret_ax = plt.gca()
    if legend:
        if isinstance(legend, Axes):
            loc = "center"
            plt.sca(legend)
            legend=(0.5,0.5)
        else:
            loc="center left"
            if not isinstance(legend, tuple) or not len(legend)==2:
                legend=(1, 0.5)
        leg = plt.legend(
            handles,
            labels,
            loc=loc,
            bbox_to_anchor=legend,
            frameon=False,
            ncol=4,
        )

        # t1 = leg.get_texts()
        # # here we create the distinct instance
        # for t in t1[1:]:
        #     t._fontproperties = t1[0]._fontproperties.copy()
        # t1[len(series)].set_size(14)
    return ret_ax

# Amount of non-linearity


In [ ]:
from matplotlib.gridspec import GridSpec
from matplotlib.layout_engine import ConstrainedLayoutEngine

def format_axes(fig):
    for i, ax in enumerate(fig.axes):
        ax.tick_params(labelbottom=False, labelleft=False)

fig1 = plt.figure(layout=ConstrainedLayoutEngine(h_pad=0.01, w_pad=0.01), figsize = (16.7/2.54,15/2.54+4))
axes1 = []
gs1 = GridSpec(6, 1, figure=fig1, width_ratios=(6,), height_ratios=(4,4,4,4,4,1), hspace=0)#############################################
axes1.append(fig1.add_subplot(gs1[0,:]))
for i in range(1,5):
    axes1.append(fig1.add_subplot(gs1[i,0]))
    axes1[i].set_aspect('auto')
axes1.append(fig1.add_subplot(gs1[5,0]))
# axes1.append(fig1.add_subplot(gs1[1:,1]))
axes1[-1].axis("off")
fig2 = plt.figure(figsize=(8.7/2.54,6/2.54))
fig3 = plt.figure(figsize=(8.7/2.54,6/2.54))

## fMRI

In [ ]:
regions = [
    10,
    30,
    50,
    70,
    100,
    150,
    200,
    230,
    270,
    300,
    350,
    400,
    450,
    500,
    550,
    600,
    650,
    700,
    750,
    800,
    850,
    900,
    950,
]
fMRI_results = {"Empirical": [], "Shadow": []}
for region in regions:
    out_name = f"clean_cra_strin_{region}_bin7"
    with open(
        os.path.join(RESULTS_FOLDER, out_name, out_name + "_globalStats.json")
    ) as fp:
        tmp = {k: np.array(v) for k, v in json.load(fp).items()}
    fMRI_results["Empirical"].append(1 - tmp["gaussMI"] / tmp["totalMI"])
    fMRI_results["Shadow"].append(1 - tmp["gaussMIshadow"] / tmp["totalMIshadow"])

In [ ]:
# fig = plt.figure(figsize=(12, 6))
boxplot(
    regions,
    fMRI_results,
    [sns.color_palette("colorblind")[1], sns.color_palette("colorblind")[0]],
    [0, 2, 5],
    # plt.gca(),
    axes1[0],
    True,
    "Bonferroni",
)
plt.sca(axes1[0])
plt.xlabel("# regions", labelpad=5)
plt.ylabel("RNL")
plt.ylim(plt.ylim()[0], 0.25)
bottom = plt.ylim()[0]
height = plt.ylim()[1] - bottom
for i in range(0, len(regions) * 2 - 2, 4):
    axes1[0].add_patch(
        Rectangle((i + 0.5, bottom), 2, height, fc=[0.85, 0.85, 0.85], zorder=0)
    )
# plt.savefig(os.path.join(FIGURES_FOLDER, "amount_fMRI.pdf"), bbox_inches="tight")
# plt.show()

## EEG

In [ ]:
EEG_results = {"Empirical": [], "Shadow": []}
for band, bin in zip(bands, nbins):
    out_name = f"EEG_124s_band_{band_names[band]}_bin{bin}"
    with open(
        os.path.join(RESULTS_FOLDER, out_name, out_name + "_globalStats.json")
    ) as fp:
        tmp = {k: np.array(v) for k, v in json.load(fp).items()}
    EEG_results["Empirical"].append(1 - tmp["gaussMI"] / tmp["totalMI"])
    EEG_results["Shadow"].append(1 - tmp["gaussMIshadow"] / tmp["totalMIshadow"])

In [ ]:
# fig = plt.figure(figsize=(12, 6))
boxplot(
    band_fancy_names,
    EEG_results,
    [sns.color_palette("colorblind")[1], sns.color_palette("colorblind")[0]],
    [0, 2, 5],
    # plt.gca(),
    axes1[1],
    True,
    "Bonferroni",
)
plt.sca(axes1[1])
# plt.xlabel("Band", labelpad=5)
axes1[1].set_xticklabels([])
plt.ylabel("EEG $-$ RNL")
plt.ylim(plt.ylim()[0], 0.25)
bottom = plt.ylim()[0]
height = plt.ylim()[1] - bottom
for i in range(0, len(bands) * 2 - 2, 4):
    axes1[1].add_patch(
        Rectangle((i + 0.5, bottom), 2, height, fc=[0.85, 0.85, 0.85], zorder=0)
    )
# plt.savefig(os.path.join(FIGURES_FOLDER, "amount_EEG.pdf"), bbox_inches="tight")
# plt.show()

## iEEG

In [ ]:
sessions = [1, "long"]
iEEG_results = {session: {"Empirical": [], "Shadow": []} for session in sessions}
for session in sessions:
    piece = session if session == "long" else f"ses{session:02}"
    for band, bin in zip(bands, nbins):
        if session == "long":
            bin = nbins[-1]
        folder_name = f"iEEG_{piece}_band_{band_names[band]}_bin{bin}"
        with open(
            os.path.join(RESULTS_FOLDER, folder_name, folder_name + "_globalStats.json")
        ) as fp:
            tmp = {k: np.array(v) for k, v in json.load(fp).items()}
        iEEG_results[session]["Empirical"].append(1 - tmp["gaussMI"] / tmp["totalMI"])
        iEEG_results[session]["Shadow"].append(
            1 - tmp["gaussMIshadow"] / tmp["totalMIshadow"]
        )

In [ ]:
# for session in [1, "long"]:
# fig = plt.figure(figsize=(12, 6))
session = 1
boxplot(
    band_fancy_names,
    iEEG_results[session],
    [sns.color_palette("colorblind")[1], sns.color_palette("colorblind")[0]],
    [0, 2, 5],
    # plt.gca(),
    axes1[3],
    True,
    "Bonferroni",
)
plt.sca(axes1[3])
# plt.xlabel("Band", labelpad=5)
# axes1[3].set_xticklabels([])
plt.ylabel("iEEG $-$ RNL")
# plt.title(f"Session: {session}")
plt.xlabel("Band", labelpad=5)
plt.ylim(plt.ylim()[0], 0.25)
bottom = plt.ylim()[0]
height = plt.ylim()[1] - bottom
for i in range(0, len(bands) * 2 - 2, 4):
    axes1[3].add_patch(
        Rectangle((i + 0.5, bottom), 2, height, fc=[0.85, 0.85, 0.85], zorder=0)
    )
# plt.savefig(
#     os.path.join(
#         FIGURES_FOLDER, f"amount_iEEG_{'short' if session==1 else session}.pdf"
#     ),
#     bbox_inches="tight",
# )
# plt.show()

## Single unit spikes

In [ ]:
GOOD_SESSIONS = [
    831882777,
    816200189,
    771160300,
    786091066,
    779839471,
    778998620,
    781842082,
    778240327,
    793224716,
    839068429,
    794812542,
    768515987,
    767871931,
    840012044,
    766640955,
    847657808,
]
groups = {}
statsReg = [{} for __ in GOOD_SESSIONS]
for s, session in enumerate(GOOD_SESSIONS):
    for folder in glob.glob(
        os.path.join(RESULTS_FOLDER, f"new_spiking_{session}_REGULAR_bin*")
    ):
        if os.path.isfile(
            os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
        ):
            with open(os.path.join(folder, "shape.json")) as fp:
                groups[session] = json.load(fp)[1]
            with open(
                os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
            ) as fp:
                tmp = {k: np.array(v) for k, v in json.load(fp).items()}
            statsReg[s] = {
                "Empirical": 1 - tmp["gaussMI"] / tmp["totalMI"],
                "Shadow": 1 - tmp["gaussMIshadow"] / tmp["totalMIshadow"],
            }

In [ ]:
sus_results = {}
for data in ["Empirical", "Shadow"]:
    sus_results[data] = [[] for i in range(6)]
    for s in range(len(GOOD_SESSIONS)):
        for i in range(6):
            sus_results[data][i].append(statsReg[s][data][i])
boxplot(
    [2**i for i in range(6)],
    sus_results,
    [sns.color_palette("colorblind")[1], sns.color_palette("colorblind")[0]],
    [0, 2, 5],
    axes1[4],
    True,
    "Bonferroni",
)
plt.sca(axes1[4])
plt.xlabel("# units per region", labelpad=5)
plt.ylabel("SUS $-$ RNL")
plt.ylim(plt.ylim()[0], 0.9)
bottom = plt.ylim()[0]
height = plt.ylim()[1] - bottom
for i in range(0, 6 * 2 - 2, 4):
    plt.gca().add_patch(
        Rectangle((i + 0.5, bottom), 2, height, fc=[0.85, 0.85, 0.85], zorder=0)
    )

# Sources of non-linearity

## EEG

In [ ]:
BLP_results = {"Empirical": [], "Shadow": [], "Naïve Shadow": []}
for band in bands:
    out_name = f"electrodeBLP_band_{band_names[band]}_bin9"
    with open(
        os.path.join(RESULTS_FOLDER, out_name, out_name + "_globalStats.json")
    ) as fp:
        tmp = {k: np.array(v) for k, v in json.load(fp).items()}
    BLP_results["Empirical"].append(1 - tmp["gaussMI"] / tmp["totalMI"])
    BLP_results["Naïve Shadow"].append(1 - tmp["gaussMIshadow"] / tmp["totalMIshadow"])
    out_name = f"electrodeBLP_shadow_band_{band_names[band]}_bin9"
    with open(
        os.path.join(RESULTS_FOLDER, out_name, out_name + "_globalStats.json")
    ) as fp:
        tmp2 = {k: np.array(v) for k, v in json.load(fp).items()}
    BLP_results["Shadow"].append(1 - tmp2["gaussMI"] / tmp2["totalMI"])

In [ ]:
# fig = plt.figure(figsize=(12, 6))
boxplot(
    band_fancy_names,
    BLP_results,
    [
        sns.color_palette("colorblind")[1],
        sns.color_palette("colorblind")[0],
        sns.color_palette("colorblind")[2],
    ],
    [0, 2, 5],
    # plt.gca(),
    axes1[2],
    ["Empirical", "Shadow"],
    "Bonferroni",
    legend=axes1[5]
)
plt.sca(axes1[2])
# plt.xlabel("Band", labelpad=5)
plt.ylabel("EEG BLP $-$ RNL")
axes1[2].set_xticklabels([])
plt.ylim(plt.ylim()[0], 0.25)
bottom = plt.ylim()[0]
height = plt.ylim()[1] - bottom
for i in range(0, len(bands) * 4 - 8, 6):
    axes1[2].add_patch(
        Rectangle((i + 1, bottom), 3, height, fc=[0.85, 0.85, 0.85], zorder=0)
    )

## iEEG

### Seizure

In [ ]:
from scipy import signal

fs_fast = 200

seizure = {}
for folder in glob.glob(os.path.join(RESULTS_FOLDER, "sampleSeizure_band*_bin*")):
    bins = int(re.findall("bin(\d+)", folder)[0])
    print(folder, bins)
    if os.path.isfile(
        os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
    ):
        band = re.findall(r"band_([a-z]+)_", folder)[0]
        with open(
            os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
        ) as fp:
            tmp = {k: np.array(v) for k, v in json.load(fp).items()}
        seizure[band] = {
            "Empirical": (tmp["totalMI"] - tmp["gaussMI"]) / tmp["totalMI"],
            "Shadow": (tmp["totalMIshadow"] - tmp["gaussMIshadow"])
            / tmp["totalMIshadow"],
        }
windowed_mean_power = np.load(
    os.path.join(MAIN_DATA_FOLDER, "iEEG", "sampleSeizure_power.npy")
)
original_iEEG = loadmat(os.path.join(MAIN_DATA_FOLDER, "iEEG", "sampleSeizure.mat"))[
    "EEG"
]
start = 1782743 - 51200
stop = 1971102 + 102400
resam, t = signal.resample(
    original_iEEG[4::8, start:stop].T,
    round((stop - start) / 1024 * fs_fast),
    np.arange(stop - start) / 1024,
)

In [ ]:
RNL = []

sec_dest = 283.936
for band in bands:
    thisRNL = seizure[band_names[band]]["Empirical"]
    N_blocks = thisRNL.shape[0]
    fs_dest = fs_band(band)
    slide_band = (fs_dest * 45) // 10
    fs_rnl = slide_band / fs_dest
    thisTime = fs_rnl * N_blocks
    tot_samp_dest = thisTime * fs_fast
    conversion_factor = tot_samp_dest / (fs_fast * fs_rnl)
    index = np.round(np.arange(tot_samp_dest) / (fs_fast * fs_rnl)).astype(int)
    index[index == thisRNL.shape[0]] = thisRNL.shape[0] - 1
    RNL.append(thisRNL[index])
tmpRNL = np.concatenate(RNL)
normalised_wmp = (windowed_mean_power - np.min(windowed_mean_power)) / (
    np.max(windowed_mean_power) - np.min(windowed_mean_power)
) * (np.nanmax(tmpRNL) - np.nanmin(tmpRNL)) + np.nanmin(tmpRNL)
N_blocks = 64
fs_dest = 1024
slide_band = (fs_dest * 45) // 10
fs_rnl = slide_band / fs_dest
thisTime = fs_rnl * N_blocks
tot_samp_dest = thisTime * fs_fast
conversion_factor = tot_samp_dest / (fs_fast * fs_rnl)
index = np.round(np.arange(tot_samp_dest) / (fs_fast * fs_rnl)).astype(int)
index[index == normalised_wmp.shape[0]] = normalised_wmp.shape[0] - 1
RNL.append(normalised_wmp[index])

In [ ]:
tot_samp = max(map(lambda x: x.shape[0], RNL))
RNL_long = np.full([tot_samp, 6], np.nan)
for b in range(6):
    RNL_long[: RNL[b].shape[0], b] = RNL[b]

In [ ]:
plt.figure(fig3.number)
plt.imshow(RNL_long.T, interpolation="nearest")
plt.gca().set_aspect(5000)

plt.xlabel("Time [s]",fontsize="large")
plt.ylabel("Band                  Electrode      ",fontsize="large")
plt.xticks(
    np.arange(12) * 5000,
    (np.arange(12) * 5000 / fs_fast).astype(int),
    rotation=30,
    ha="right",
    rotation_mode="anchor",fontsize="large"
)
plt.yticks(
    np.arange(6),
    ["power\n(a.u.)", r"$\gamma$", r"$\beta$", r"$\alpha$", r"$\theta$", r"$\delta$"][
        ::-1
    ],fontsize="large",
)
# plt.ylim(plt.ylim())
plt.plot(t * 200, resam / 2000 + np.arange(7) + 6, "k", lw=0.5, alpha=0.5)
plt.ylim((-0.5, 13))
plt.vlines(
    [50 * fs_fast, (183.936 + 50) * fs_fast],
    *plt.ylim(),
    colors=sns.color_palette("colorblind")[2],
    linestyles="solid",
    lw=2,
    zorder=20
)
plt.vlines(
    [(50 - 45) * fs_fast, (183.936 + 50 - 45) * fs_fast],
    -0.5,
    5.5,
    colors=sns.color_palette("colorblind")[0],
    linestyles="dashed",
    lw=2,
    zorder=20,
)
plt.colorbar(shrink=0.7).ax.set_ylabel("RNL", rotation=270, labelpad=15)
plt.xlim((0, 283.936 * fs_fast))
# plt.savefig(
#     os.path.join(FIGURES_FOLDER, "sources_iEEG_seizure.pdf"), bbox_inches="tight"
# )
# plt.show()
plt.figure(fig1.number)

for ax, l in zip(axes1, "ABCDE"):
    ax.text(-0.01, 1, f"{l}", horizontalalignment="right", verticalalignment="center_baseline", fontweight="bold", transform=ax.transAxes)
plt.savefig(os.path.join(FIGURES_FOLDER, f"amounts.pdf"), bbox_inches="tight",)
plt.figure(fig3.number)
plt.savefig(os.path.join(FIGURES_FOLDER, f"seizure.pdf"), bbox_inches="tight",)
plt.figure(fig2.number)

## Single unit spikes

### Epochs

In [ ]:
GOOD_SESSIONS = [
    831882777,
    816200189,
    771160300,
    786091066,
    779839471,
    778998620,
    781842082,
    778240327,
    793224716,
    839068429,
    794812542,
    768515987,
    767871931,
    840012044,
    766640955,
    847657808,
]
groups = {}
statsRegSpl = [{} for __ in GOOD_SESSIONS]
for s, session in enumerate(GOOD_SESSIONS):
    for folder in glob.glob(os.path.join(RESULTS_FOLDER, f"new_spiking_{session}_REGULAR_*")):
        if "spl" in folder:
            matched = re.findall(f"_spl(\d+)_part(\d+)_", folder)[0]
            spl = int(matched[0])
            part = int(matched[1])
        else:
            spl = 1
            part = 0
        if os.path.isfile(
            os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
        ):
            with open(os.path.join(folder, "shape.json")) as fp:
                groups[session] = json.load(fp)[1]
            with open(
                os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
            ) as fp:
                tmp = {k: np.array(v) for k, v in json.load(fp).items()}
            statsRegSpl[s][(spl, part)] = {
                "Empirical": 1 - tmp["gaussMI"] / tmp["totalMI"],
                "Shadow": 1 - tmp["gaussMIshadow"] / tmp["totalMIshadow"],
            }

In [ ]:
avgRNL = np.zeros((6, 1, 6))
for s, spl in enumerate([1, 2, 3, 5, 6, 10]):
    tmp = np.zeros((1, 6))
    try:
        for p in range(spl):
            tmp[0, :] += statsRegSpl[0][(spl, p)]["Empirical"]
    except:
        continue
    avgRNL[s, :, :] += tmp/ spl

In [ ]:
vmax = np.max(avgRNL)
vmin = np.min(avgRNL)
plt.figure(fig2.number)
plt.imshow(avgRNL[:, 0, :].T, vmax=vmax, vmin=vmin)
xlabels = [f"{30//s}" for s in [1, 2, 3, 5, 6, 10]]
ylabels = [2**i for i in range(6)]
plt.xticks(np.arange(6), xlabels, rotation=45, ha="right", rotation_mode="anchor")
plt.yticks(np.arange(6), ylabels)
plt.ylabel("# units per site",fontsize="large")
plt.xlabel(f"Epoch duration [min]")
cbar = plt.colorbar(shrink=0.6, orientation='vertical', aspect=10)
cbar.ax.set_xlabel("RNL")#, rotation=90)
cbar.ax.get_yaxis().labelpad = 15
plt.savefig(os.path.join(FIGURES_FOLDER, f"sus_epochs.pdf"), bbox_inches="tight",)
plt.show()

# Localisation of non-linearities


In [ ]:
from s03a_localisation import (
    compute_localised_non_linearity,
    show_localised_non_linearity,
    plot_summary,
)

## fMRI

In [ ]:
pair_num = 4005
subj_num = 245
regions_num = 90
cut_pos = (-15, -75, 27)
results_file = os.path.join(
    RESULTS_FOLDER, "eso245_aal_{0}_90_bin{1}/subject{2:02}_{1}.npy"
)
subset_description = "fMRI AAL90 "
subset_identifiers = ["strin", "mod", "raw"]  # ["raw"]#
subset_names = ["Stringent", "Moderate", "Raw"]  # ["strin"]#
output_prefix = "fMRI_AAL90"
nbins = 7
with open(
    os.path.join(os.path.dirname(results_file.format("strin", nbins, 0)), "shape.json")
) as fp:
    samples = json.load(fp)[0]
compute_localised_non_linearity(
    results_file,
    subset_identifiers,
    output_prefix,
    pair_num,
    subj_num,
    nbins,
    samples,
)
results_file = os.path.join(
    RESULTS_FOLDER, "eso245_aal_{0}_90_bin{1}/subject{2:02}_{1}_sha.npy"
)
compute_localised_non_linearity(
    results_file,
    subset_identifiers,
    output_prefix + "_sha",
    pair_num,
    subj_num,
    nbins,
    samples,
)
show_localised_non_linearity(
    results_file,
    subset_description,
    subset_identifiers,
    subset_names,
    output_prefix,
    pair_num,
    regions_num,
    FIGURES_FOLDER,
    cut_pos,
)

## EEG

In [ ]:
pair_num = 1431#18915
subj_num = 150#50
elec_num = 54#195
results_file = os.path.join(
    RESULTS_FOLDER, "EEG_124s_band_{0}_bin{1}/subject{2:02}_{1}.npy"
)
subset_description = "EEG "
subset_identifiers = ["delta", "theta", "alpha", "beta", "gamma"]
subset_names = [r"$\delta$", r"$\theta$", r"$\alpha$", r"$\beta$", r"$\gamma$"]
output_prefix = "EEG_54"
nbins = [10, 13, 14, 20, 23]
samples = []
for band, bins in zip(subset_identifiers, nbins):
    with open(
        os.path.join(os.path.dirname(results_file.format(band, bins, 0)), "shape.json")
    ) as fp:
        samples.append(json.load(fp)[0])

compute_localised_non_linearity(
    results_file,
    subset_identifiers,
    output_prefix,
    pair_num,
    subj_num,
    nbins,
    samples,
)

results_file = os.path.join(
    RESULTS_FOLDER, "EEG_124s_band_{0}_bin{1}/subject{2:02}_{1}_sha.npy"
)
compute_localised_non_linearity(
    results_file,
    subset_identifiers,
    output_prefix + "_sha",
    pair_num,
    subj_num,
    nbins,
    samples,
)

show_localised_non_linearity(
    results_file,
    subset_description,
    subset_identifiers,
    subset_names,
    output_prefix,
    pair_num,
    elec_num,
    FIGURES_FOLDER,
)

In [ ]:
subset_names = [r"$\delta$", r"$\theta$", r"$\alpha$", r"$\beta$", r"$\gamma$"]
output_prefix = "EEG_54"
plot_summary(subset_names, output_prefix, FIGURES_FOLDER, is_aal=False)

In [ ]:
assert False

# Reliability of non-linearity
## fMRI test-retest

In [ ]:
for corkind, start, exp, sentence in [("cor",0,2, "with ranking from TMI"),("cor",1,2, "across sessions (GMI)"),("cor",1,1, "across sessions (corr)")]:#("spe",1,1, "across sessions")

    corrector = Corrector(
        bins=8,
        duration=657,
        steps=200,
        iterations=50000,
        samples=657,
        folder_name=os.path.join(RESULTS_FOLDER, "lemon_aal_strin_90_ses-01_bin8"),
        cache_dir=cache_dir,
        verbose=True,
    )
    corrector.compute_correction()
    all_cov = np.empty([14, 6, 4005])
    for subject in range(14):
        for session in range(3):
            all_cov[subject, 2 * session, :] = corrector.correct(
                np.load(
                    os.path.join(
                        RESULTS_FOLDER,
                        f"lemon_aal_strin_90_ses-0{session}_bin8/subject{subject:02}_8.npy",
                    )
                )[:, 0]
            )
            all_cov[subject, 2 * session + 1, :] = np.load(
                    os.path.join(
                        RESULTS_FOLDER,
                        f"lemon_aal_strin_90_ses-0{session}_bin8/subject{subject:02}_8_{corkind}.npy",
                    )
                )**exp
            
    pecorr = np.empty([6, 6, 4005])
    for c in range(4005):
        pecorr[:, :, c] = np.corrcoef(
            np.argsort(np.argsort(all_cov[:, :, c], axis=0), axis=0), rowvar=False
        )
    cov = np.mean(pecorr, axis=2)
    np.fill_diagonal(cov, np.nan)

    tmi = pecorr[::2,::2,:]
    spe = pecorr[1::2,start::2,:]
    for i in range(3):
        tmi[i,i,:]=np.nan
        spe[i,i,:]=np.nan
    tmi_f = tmi.flatten()
    tmi_f = tmi_f[np.isfinite(tmi_f)]
    spe_f = spe.flatten()
    spe_f = spe_f[np.isfinite(spe_f)]
    print(f"Correlation {sentence}\nTMI: {np.nanmean(tmi):.3}+/-{np.nanstd(tmi):.3},\n{corkind}: {np.nanmean(spe):.3}+/-{np.nanstd(spe):.3},\nt-test: {ttest_rel(tmi_f, spe_f,nan_policy='raise', alternative='two-sided')}")

    plt.figure(figsize=(5,5))
    plt.imshow(cov)
    plt.colorbar()
    plt.yticks(
        range(6),
        labels=[f"{i+1} ses - {j}" for i in range(3) for j in ["TMI", r"$I_{COR}$"]],
    )
    plt.xticks(
        range(6),
        labels=[f"{i+1} ses - {j}" for i in range(3) for j in ["TMI", r"$I_{COR}$"]],
        rotation=90,
    )
    plt.title("Average across 4005 pairs", pad=15)
    plt.gca().set_facecolor((0.85, 0.85, 0.85))
    plt.savefig(
        os.path.join(FIGURES_FOLDER, f"reliability_fMRI_matrix_subjectRank_{start}_{exp}.pdf"),
        bbox_inches="tight",
    )
    plt.show()

    plt.figure(figsize=(5,3.5))
    differences = np.diff(cov, axis=0)[::2, :]
    max_abs = np.nanmax(np.abs(differences))
    norm = TwoSlopeNorm(vcenter=0, vmax=max_abs, vmin=-max_abs)
    # plt.imshow(mean_all_pair, cmap=plt.cm.RdYlBu_r, norm=norm)
    plt.imshow(differences, cmap=plt.cm.RdYlBu_r, norm=norm)
    plt.colorbar(shrink=0.7).ax.set_ylabel(
        "Correlation improvement\nusing Gaussian estimate", fontsize=12
    )
    plt.yticks(range(3), labels=[f"{i+1} ses" for i in range(3)])  # , fontsize=15)
    plt.xticks(
        range(6),
        labels=[f"{i+1} ses - {j}" for i in range(3) for j in ["TMI", r"$I_{COR}$"]],
        rotation=20,
        ha="right",
        rotation_mode="anchor",
        # fontsize=15,
    )
    plt.ylabel("Predictors from:")  # , fontsize=16)
    plt.xlabel("Predicting:")  # , fontsize=16)
    plt.title("Average across 4005 pairs", pad=15)  # , fontsize=18)
    plt.gca().set_facecolor((0.85, 0.85, 0.85))
    plt.savefig(
        os.path.join(FIGURES_FOLDER, f"reliability_fMRI_improvement_subjectRank_{start}_{exp}.pdf"),
        bbox_inches="tight",
    )
    plt.show()

# Supporting information

## EEG

In [ ]:
tot_samples = [1116, 2232, 3348, 8432, 12276]
for band, bin, samples in zip(bands, nbins, tot_samples):
    for corkind, start, exp, sentence in [("cor",0,2, "with ranking from TMI"),("spe",1,1, "across sessions")]:

        corrector = Corrector(
            bins=bin,
            duration=samples,
            steps=200,
            iterations=50000,
            samples=samples,
            folder_name=os.path.join(RESULTS_FOLDER, f"EEG_124s_band_{band_names[band]}_bin{bin}"),
            cache_dir=cache_dir,
            verbose=True,
        )
        corrector.compute_correction()
        all_cov = np.empty([150, 6, 1431])
        for subject in range(150):
            for session in range(3):
                sespart="" if session==0 else f"_ses{session+1}"
                all_cov[subject, 2 * session, :] = corrector.correct(
                    np.load(
                        os.path.join(
                            RESULTS_FOLDER,
                            f"EEG_124s_band_{band_names[band]}{sespart}_bin{bin}/subject{subject:02}_{bin}.npy",
                        )
                    )[:, 0]
                )
                all_cov[subject, 2 * session + 1, :] = np.load(
                        os.path.join(
                            RESULTS_FOLDER,
                            f"EEG_124s_band_{band_names[band]}{sespart}_bin{bin}/subject{subject:02}_{bin}_{corkind}.npy",
                        )
                    )**exp
                
        pecorr = np.empty([6, 6, 1431])
        for c in range(1431):
            pecorr[:, :, c] = np.corrcoef(
                np.argsort(np.argsort(all_cov[:, :, c], axis=0), axis=0), rowvar=False
            )
        cov = np.mean(pecorr, axis=2)
        np.fill_diagonal(cov, np.nan)

        tmi = pecorr[::2,::2,:]
        spe = pecorr[1::2,start::2,:]
        for i in range(3):
            tmi[i,i,:]=np.nan
            spe[i,i,:]=np.nan
        tmi_f = tmi.flatten()
        tmi_f = tmi_f[np.isfinite(tmi_f)]
        spe_f = spe.flatten()
        spe_f = spe_f[np.isfinite(spe_f)]
        print(f"Band {band_names[band]} - Correlation {sentence}\nTMI: {np.nanmean(tmi):.3}+/-{np.nanstd(tmi):.3},\n{corkind}: {np.nanmean(spe):.3}+/-{np.nanstd(spe):.3},\nt-test: {ttest_rel(tmi_f, spe_f,nan_policy='raise', alternative='less')}")

        plt.hist(spe_f-tmi_f, bins=50)
        plt.title(f"{np.mean(spe_f-tmi_f)} {len(spe_f)}")
        plt.show()
        plt.figure(figsize=(5,5))
        plt.imshow(cov)
        plt.colorbar()
        plt.yticks(
            range(6),
            labels=[f"{i+1} ses - {j}" for i in range(3) for j in ["TMI", r"$I_{COR}$"]],
        )
        plt.xticks(
            range(6),
            labels=[f"{i+1} ses - {j}" for i in range(3) for j in ["TMI", r"$I_{COR}$"]],
            rotation=90,
        )
        plt.title(f"Band {band_fancy_names[band]} - Average across 1431 pairs", pad=15)
        plt.gca().set_facecolor((0.85, 0.85, 0.85))
        plt.savefig(
            os.path.join(FIGURES_FOLDER, f"reliability_fMRI_matrix_subjectRank.pdf"),
            bbox_inches="tight",
        )
        plt.show()

        plt.figure(figsize=(5,3.5))
        differences = np.diff(cov, axis=0)[::2, :]
        max_abs = np.nanmax(np.abs(differences))
        norm = TwoSlopeNorm(vcenter=0, vmax=max_abs, vmin=-max_abs)
        # plt.imshow(mean_all_pair, cmap=plt.cm.RdYlBu_r, norm=norm)
        plt.imshow(differences, cmap=plt.cm.RdYlBu_r, norm=norm)
        plt.colorbar(shrink=0.7).ax.set_ylabel(
            "Correlation improvement\nusing Gaussian estimate", fontsize=16
        )
        plt.yticks(range(3), labels=[f"{i+1} ses" for i in range(3)])  # , fontsize=15)
        plt.xticks(
            range(6),
            labels=[f"{i+1} ses - {j}" for i in range(3) for j in ["TMI", r"$I_{COR}$"]],
            rotation=20,
            ha="right",
            rotation_mode="anchor",
            # fontsize=15,
        )
        plt.ylabel("Predictors from:")  # , fontsize=16)
        plt.xlabel("Predicting:")  # , fontsize=16)
        plt.title(f"Band {band_fancy_names[band]} - Average across 1431 pairs", pad=15)  # , fontsize=18)
        plt.gca().set_facecolor((0.85, 0.85, 0.85))
        plt.savefig(
            os.path.join(FIGURES_FOLDER, f"reliability_fMRI_improvement_subjectRank.pdf"),
            bbox_inches="tight",
        )
        plt.show()

## iEEG

In [ ]:
session = "long"
fig4 = plt.figure(figsize=(11.4/2.54,10/2.54))
boxplot(
    band_fancy_names,
    iEEG_results[session],
    [sns.color_palette("colorblind")[1], sns.color_palette("colorblind")[0]],
    [0, 2, 5],
    plt.gca(),
    # axes1[2],
    True,
    "Bonferroni",True
)
# plt.xlabel("Band", labelpad=5)
# plt.gca().set_xticklabels([])
plt.ylabel("RNL")
# plt.title(f"Session: {session}")
plt.ylim(plt.ylim()[0], 0.35)
bottom = plt.ylim()[0]
height = plt.ylim()[1] - bottom
for i in range(0, len(bands) * 2 - 2, 4):
    plt.gca().add_patch(
        Rectangle((i + 0.5, bottom), 2, height, fc=[0.85, 0.85, 0.85], zorder=0)
    )
plt.savefig(os.path.join(FIGURES_FOLDER, f"iEEG_long.pdf"), bbox_inches="tight",)
plt.show()

In [ ]:
tot_samples = [1116, 2232, 3348, 8432, 12276]
for band, bin, samples in zip(bands, nbins, tot_samples):
    for corkind, start, exp, sentence in [("cor",0,2, "with ranking from TMI"),("spe",1,1, "across sessions")]:

        corrector = Corrector(
            bins=bin,
            duration=samples,
            steps=200,
            iterations=50000,
            samples=samples,
            folder_name=os.path.join(RESULTS_FOLDER, f"iEEG_ses01_band_{band_names[band]}_bin{bin}"),
            cache_dir=cache_dir,
            verbose=True,
        )
        corrector.compute_correction()
        all_cov = np.empty([18, 6, 276])
        for subject in range(18):
            for session in range(3):
                sespart=f"{session+1:02}"
                all_cov[subject, 2 * session, :] = corrector.correct(
                    np.load(
                        os.path.join(
                            RESULTS_FOLDER,
                            f"iEEG_ses{sespart}_band_{band_names[band]}_bin{bin}/subject{subject:02}_{bin}.npy",
                        )
                    )[:, 0]
                )
                all_cov[subject, 2 * session + 1, :] = np.load(
                        os.path.join(
                            RESULTS_FOLDER,
                            f"iEEG_ses{sespart}_band_{band_names[band]}_bin{bin}/subject{subject:02}_{bin}_{corkind}.npy",
                        )
                    )**exp
                
        pecorr = np.empty([6, 6, 276])
        for c in range(276):
            pecorr[:, :, c] = np.corrcoef(
                np.argsort(np.argsort(all_cov[:, :, c], axis=0), axis=0), rowvar=False
            )
        cov = np.mean(pecorr, axis=2)
        np.fill_diagonal(cov, np.nan)

        tmi = pecorr[::2,::2,:]
        spe = pecorr[1::2,start::2,:]
        for i in range(3):
            tmi[i,i,:]=np.nan
            spe[i,i,:]=np.nan
        tmi_f = tmi.flatten()
        tmi_f = tmi_f[np.isfinite(tmi_f)]
        spe_f = spe.flatten()
        spe_f = spe_f[np.isfinite(spe_f)]
        print(f"Band {band_names[band]} - Correlation {sentence}\nTMI: {np.nanmean(tmi):.3}+/-{np.nanstd(tmi):.3},\n{corkind}: {np.nanmean(spe):.3}+/-{np.nanstd(spe):.3},\nt-test: {ttest_rel(tmi_f, spe_f,nan_policy='raise', alternative='less')}")

        plt.figure(figsize=(5,5))
        plt.imshow(cov)
        plt.colorbar()
        plt.yticks(
            range(6),
            labels=[f"{i+1} ses - {j}" for i in range(3) for j in ["TMI", r"$I_{COR}$"]],
        )
        plt.xticks(
            range(6),
            labels=[f"{i+1} ses - {j}" for i in range(3) for j in ["TMI", r"$I_{COR}$"]],
            rotation=90,
        )
        plt.title(f"Band {band_fancy_names[band]} - Average across 276 pairs", pad=15)
        plt.gca().set_facecolor((0.85, 0.85, 0.85))
        plt.savefig(
            os.path.join(FIGURES_FOLDER, f"reliability_fMRI_matrix_subjectRank.pdf"),
            bbox_inches="tight",
        )
        plt.show()

        plt.figure(figsize=(5,3.5))
        differences = np.diff(cov, axis=0)[::2, :]
        max_abs = np.nanmax(np.abs(differences))
        norm = TwoSlopeNorm(vcenter=0, vmax=max_abs, vmin=-max_abs)
        # plt.imshow(mean_all_pair, cmap=plt.cm.RdYlBu_r, norm=norm)
        plt.imshow(differences, cmap=plt.cm.RdYlBu_r, norm=norm)
        plt.colorbar(shrink=0.7).ax.set_ylabel(
            "Correlation improvement\nusing Gaussian estimate", fontsize=16
        )
        plt.yticks(range(3), labels=[f"{i+1} ses" for i in range(3)])  # , fontsize=15)
        plt.xticks(
            range(6),
            labels=[f"{i+1} ses - {j}" for i in range(3) for j in ["TMI", r"$I_{COR}$"]],
            rotation=20,
            ha="right",
            rotation_mode="anchor",
            # fontsize=15,
        )
        plt.ylabel("Predictors from:")  # , fontsize=16)
        plt.xlabel("Predicting:")  # , fontsize=16)
        plt.title(f"Band {band_fancy_names[band]} - Average across 276 pairs", pad=15)  # , fontsize=18)
        plt.gca().set_facecolor((0.85, 0.85, 0.85))
        plt.savefig(
            os.path.join(FIGURES_FOLDER, f"reliability_fMRI_improvement_subjectRank.pdf"),
            bbox_inches="tight",
        )
        plt.show()

### Other Activity

In [ ]:
with open(
    os.path.join(MAIN_DATA_FOLDER, "iEEG", "spikes_by_subject_electrode.json")
) as fp:
    spikes_per_electrode = {
        int(k): np.array([len(v[a][0]) for a in v]) for k, v in json.load(fp).items()
    }
fs_dest = [fs_band(band) for band in bands]

In [ ]:
fig = plt.figure(figsize=(11.4/2.54,10/2.54))
spikes_per_subject = [
    np.sum(spikes_per_electrode[k]) for k in sorted(spikes_per_electrode.keys())
]
for band, (rnl, fmt) in enumerate(zip(iEEG_results[1]["Empirical"], "+^s3o"), start=1):
    r = pearsonr(spikes_per_subject, rnl)
    label = (
        f"{band_fancy_names[band]} - r:{r.statistic:.3} {('*' if r.pvalue > 0.002 else ('**' if r.pvalue > 0.0002 else ('***' if r.pvalue > 0.00002 else '****'))) if r.pvalue<0.01 else ''}"
    )
    print(band_fancy_names[band],r.pvalue*5)
    p = plt.scatter(spikes_per_subject, rnl, marker=fmt, label=label)
    par = np.polyfit(spikes_per_subject, rnl, 1)
    plt.xlim(plt.xlim())
    plt.plot(
        plt.xlim(),
        np.polyval(par, plt.xlim()),
        color=p.get_facecolor(),
        lw=2 if r.pvalue < 0.01 else 0.8,
        ls="-" if r.pvalue < 0.01 else ":",
    )
plt.xlabel("#IED")
plt.ylabel("RNL")
plt.legend(loc="center left", title="Band", bbox_to_anchor=(1, 0.5), frameon=False)
plt.savefig(os.path.join(FIGURES_FOLDER, "sources_iEEG_IED.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
def data_from(band, nbins, duration):
    MI_thres = {10: 0.015696, 13: 0.009054, 14: 0.008483, 20: 0.003671, 23: 0.002931}
    corrct = Corrector(
        nbins,
        duration,
        folder_name=os.path.join(RESULTS_FOLDER, f"iEEG_ses01_band_{band_names[band]}_bin{nbins}"),
        config="../config.ini",
        cache_dir="../cache/",
        display=False,
        verbose=True,
    )
    corrct.compute_correction()
    ns = []
    rnl = []
    aga = []
    for subj in range(18):
        MI = corrct.correct(
            np.load(
                os.path.join(
                    RESULTS_FOLDER,
                    f"iEEG_ses01_band_{band_names[band]}_bin{nbins}",
                    f"subject{subj:02}_{nbins}.npy",
                )
            )
        )
        tr = MI[:, 0]
        ga = np.mean(MI[:, 1:], axis=1)
        rnl.append(1 - ga / tr)
        aga.append(ga)
        fe, se = np.triu_indices(24, 1)
        ns.append(
            spikes_per_electrode[subj + 1][fe]
            + spikes_per_electrode[subj + 1][se]  # [band-1][band-1]
        )
    ns = np.concatenate(ns)
    aga = np.concatenate(aga)
    rnl = np.concatenate(rnl)
    good = (rnl >= 0) & np.isfinite(rnl) & (aga > MI_thres[nbins])
    return ns, rnl, good

In [ ]:
data = [
    data_from(band, nbins, duration)
    for band, (duration, nbins) in enumerate(  # 23, 12276
        [(1116, 10), (2232, 13), (3348, 14), (8432, 20), (12276, 23)], start=1
    )
]
plt.close()
fig, ax = plt.subplots(2, 3, figsize=(15, 10), sharex=None, sharey="all")
plt.subplots_adjust(hspace=0.13, wspace=0)
for band, (ns, rnl, good) in enumerate(data, start=1):
    print(np.sum(rnl > 1))
    good = np.isfinite(rnl) & (rnl > -5)
    his = ax[(band - 1) // 3, (band - 1) % 3].hist2d(
        ns[good],
        rnl[good],
        norm=LogNorm(),
        bins=[np.arange(11) * 25, np.arange(13) / 2 - 5],
    )
    r = pearsonr(ns[good], rnl[good])
    ax[(band - 1) // 3, (band - 1) % 3].set_title(
        f"{band_names[band]} - r:{r.statistic:.3} {('*' if r.pvalue > 0.002 else ('**' if r.pvalue > 0.0002 else ('***' if r.pvalue > 0.00002 else '****'))) if r.pvalue<0.01 else ''}",
        y=1.0,
        pad=5,
    )
    if (band - 1) // 3:
        ax[(band - 1) // 3, (band - 1) % 3].set_xlabel("# IED")
    if not (band - 1) % 3:
        ax[(band - 1) // 3, (band - 1) % 3].set_ylabel("RNL")
plt.colorbar(his[-1], ax=ax[1, 2]).set_label("# pairs")
ax[1, 2].axis("off")
plt.savefig(
    os.path.join(FIGURES_FOLDER, "sources_iEEG_IED_all.pdf"), bbox_inches="tight"
)
plt.show()
fig, ax = plt.subplots(2, 3, figsize=(15, 10), sharex=None, sharey="all")
plt.subplots_adjust(hspace=0.13, wspace=0)
for band, (ns, rnl, good) in enumerate(data, start=1):
    his = ax[(band - 1) // 3, (band - 1) % 3].hist2d(
        ns[good], rnl[good], norm=LogNorm()
    )
    r = pearsonr(ns[good], rnl[good])
    ax[(band - 1) // 3, (band - 1) % 3].set_title(
        f"{band_names[band]} - r:{r.statistic:.3} {('*' if r.pvalue > 0.002 else ('**' if r.pvalue > 0.0002 else ('***' if r.pvalue > 0.00002 else '****'))) if r.pvalue<0.01 else ''}",
        y=1.0,
        pad=5,
    )
    if (band - 1) // 3:
        ax[(band - 1) // 3, (band - 1) % 3].set_xlabel("# IED")
    if not (band - 1) % 3:
        ax[(band - 1) // 3, (band - 1) % 3].set_ylabel("RNL")
plt.colorbar(his[-1], ax=ax[1, 2]).set_label("# pairs")
ax[1, 2].axis("off")
plt.ylim(bottom=0, top=1)
plt.savefig(
    os.path.join(FIGURES_FOLDER, "sources_iEEG_IED_significantPairs.pdf"),
    bbox_inches="tight",
)
plt.show()

# Illustrative

In [ ]:
from scipy.stats import chi
from numpy.random.mtrand import RandomState as RandomState
from mienc.support import pair_mutual_information, surrogate, normalise

class get_ellipsis():
    def __init__(self, rho, df):
        self.distribution = chi(df)
        self.rho = rho
        self.b = np.sqrt((1 - rho) / (1 + rho))

    def get_sample(self, size: tuple, random_state: RandomState) -> np.ndarray:
        samples_number = np.prod(size)
        exp = np.zeros((samples_number, 2))
        exp[:, 0] = self.distribution.rvs(samples_number, random_state=random_state)
        return np.reshape(
            self._rotation(
                self._rotation(
                    exp, 2 * np.pi * random_state.random(samples_number), self.b
                ),
                np.pi / 4,
            ),
            [*size, 2],
        )

    def _rot_mat(self, theta, B):
        _theta = np.array(theta)
        return np.moveaxis(
            np.array(
                [
                    [np.cos(_theta), B * np.sin(_theta)],
                    [-np.sin(_theta) / B, np.cos(_theta)],
                ]
            ),
            2,
            0,
        )

    def _rotation(self, vec: np.ndarray, theta: float, B: float = 1):
        """
        Rotates vector on the plane of an angle theta around an ellipsis with y-oriented axis B times the x-oriented one.
        Input are expected to be np.ndarrays of any shape as long as at leas one dimension ha length two. If more than one
        dimension has length 2, the first is considered to be the spatial dimension.
        """
        if 2 in vec.shape:
            myAx = np.min(np.nonzero(np.array(vec.shape) == 2))
            _vec = np.moveaxis(vec, myAx, 0)
            myShape = _vec.shape
            _vec = _vec.reshape((2, -1))
            if np.isscalar(theta):
                rot_mat = self._rot_mat(np.full(_vec.shape[1], theta), B)
            else:
                rot_mat = self._rot_mat(theta, B)
            res = np.empty_like(_vec)
            for i in range(_vec.shape[1]):
                res[:, i] = _vec[:, i] @ rot_mat[i]
            return np.moveaxis(res.reshape(myShape), 0, myAx)
        else:
            raise ValueError(f"Invalid input shape {vec.shape}")

In [ ]:
fig, ax = plt.subplots(1,4, figsize=(8.7/2.54,6/2.54), sharex=True, sharey=True)
plt.subplots_adjust(hspace=0.13, wspace=0)
rs=np.random.default_rng(5)
corrector = Corrector(
    bins=7,
    duration=400,
    steps=200,
    iterations=10000,
    samples=400,
    folder_name="",
    cache_dir=cache_dir,
    verbose=True,
    workers=48
)
corrector.compute_correction()
for i, (r, d) in enumerate([(0.5,100), (0.5,6), (0.5,2.5), (0.5,2)]):
    x = get_ellipsis(r,d).get_sample((400,), rs)
    x/=x.max()
    ax[i].scatter(x[:,0], x[:,1], s=1)
    ax[i].set_aspect('equal', adjustable="box")
    ax[i].set_xticks([])
    ax[i].set_yticks([])
    MI=corrector.correct(pair_mutual_information(x[:,0], x[:,1],7))
    surr = np.empty(99)
    x[:,0]= normalise(x[:,0])
    x[:,1]= normalise(x[:,1])
    for j in range(99):
        y = surrogate(x,True,1, rs)
        surr[j] = pair_mutual_information(y[:,0], y[:,1],7)
    ax[i].set_xlabel(f"RNL: {1-corrector.correct(surr).mean()/MI:.2}", fontsize=8)
    if i==2:
        A=1-surr.mean()/MI
    if i==3:
        B=1-surr.mean()/MI
plt.savefig(
    os.path.join(FIGURES_FOLDER, "sample_distributions.pdf"),
    bbox_inches="tight",
)
plt.show()